# SONORA Stage 5 — AudioSep Context / Chunking Ablation

**Purpose:** benchmark-only experiment. This notebook does not edit SONORA application code and does not download model weights. It reuses the pinned AudioSep checkpoint already stored in Google Drive, verifies its SHA-256, and compares direct inference with chunked inference under multiple context/core settings.

## Important safeguards
- Run in a fresh disposable Colab GPU runtime (T4 or better recommended).
- No `pip install`, no dependency upgrades, no global pickle override.
- Uses the existing AudioSep checkpoint in `MyDrive/SONORA_Stage5/checkpoints/audiosep/`.
- The checkpoint is loaded only after SHA-256 verification. The legacy upstream loader is wrapped so unrestricted pickle fallback is allowed only for the two exact, verified checkpoint paths, then `torch.load` is restored.
- Upload a representative instrument recording when prompted. If none is uploaded, the notebook uses a clearly labelled synthetic test signal; that cannot establish musical quality.
- The direct full-length output is a comparison reference, not clean ground truth. Correlation alone is not a perceptual-quality score.
- Outputs, CSV, JSON and plots are saved under `MyDrive/SONORA_Stage5/context_sweep/`.


In [ ]:
import os, sys, json, hashlib, shutil, subprocess, time, gc, platform
from pathlib import Path
import numpy as np
import torch
import soundfile as sf
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from google.colab import drive, files

assert torch.cuda.is_available(), 'GPU unavailable. Select Runtime > Change runtime type > T4 GPU, then rerun.'
print('Python:', platform.python_version(), '| Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/SONORA_Stage5')
CKPT_DIR=ROOT/'checkpoints'/'audiosep'
MODEL_CKPT=CKPT_DIR/'audiosep_base_4M_steps.ckpt'
CLAP_CKPT=CKPT_DIR/'music_speech_audioset_epoch_15_esc_89.98.pt'
OUT=ROOT/'context_sweep'; OUT.mkdir(parents=True,exist_ok=True)
assert MODEL_CKPT.is_file() and CLAP_CKPT.is_file(), f'Missing saved AudioSep checkpoints in {CKPT_DIR}. Do not download here; restore the verified files from your previous benchmark first.'
EXPECTED={MODEL_CKPT:'f8cda01bfd0ebd141eef45d41db7a3ada23a56568465840d3cff04b8010ce82c', CLAP_CKPT:'51c68f12f9d7ea25fdaaccf741ec7f81e93ee594455410f3bca4f47f88d8e006'}
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
for p,want in EXPECTED.items():
    got=sha256(p); print(p.name, p.stat().st_size, got)
    assert got==want, f'Checkpoint SHA-256 mismatch: {p}'
print('[OK] Both existing checkpoints verified. No weights downloaded.')


In [ ]:
REPO=Path('/content/AudioSep')
PIN='944583f18b84589dc965de3ad77525c945334252'
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/Audio-AGI/AudioSep.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--all','--tags'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',PIN],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==PIN, (head,PIN)
print('[OK] AudioSep source pinned:',head)
LOCAL_CLAP_CKPT=REPO/'checkpoint'/CLAP_CKPT.name
LOCAL_CLAP_CKPT.parent.mkdir(parents=True,exist_ok=True)
if not LOCAL_CLAP_CKPT.exists() or sha256(LOCAL_CLAP_CKPT)!=EXPECTED[CLAP_CKPT]: shutil.copy2(CLAP_CKPT,LOCAL_CLAP_CKPT)
assert sha256(LOCAL_CLAP_CKPT)==EXPECTED[CLAP_CKPT]
print('[OK] CLAP checkpoint staged at upstream expected path:',LOCAL_CLAP_CKPT)


In [ ]:
import contextlib
VERIFIED={p.resolve() for p in EXPECTED}
AUDIT=[]
@contextlib.contextmanager
def scoped_verified_torch_load(allowed):
    allowed={Path(p).resolve() for p in allowed}
    assert allowed <= VERIFIED
    original=torch.load
    def guarded(f,*args,**kwargs):
        raw=f if isinstance(f,(str,os.PathLike)) else getattr(f,'name',None)
        target=Path(raw).resolve() if raw else None
        if kwargs.get('weights_only') is not None:
            if kwargs['weights_only'] is False and target not in allowed: raise PermissionError(f'Blocked unsafe load: {raw}')
            return original(f,*args,**kwargs)
        clean={k:v for k,v in kwargs.items() if k!='weights_only'}
        pos=f.tell() if hasattr(f,'tell') and hasattr(f,'seek') else None
        try:
            obj=original(f,*args,weights_only=True,**clean); AUDIT.append({'path':str(target or raw),'mode':'weights_only=True'}); return obj
        except Exception as exc:
            if target not in allowed: raise PermissionError(f'Blocked pickle fallback for {raw}') from exc
            if pos is not None: f.seek(pos)
            AUDIT.append({'path':str(target),'mode':f'weights_only=False scoped fallback ({type(exc).__name__})'})
            return original(f,*args,weights_only=False,**clean)
    torch.load=guarded
    try: yield
    finally: torch.load=original

os.chdir(REPO)
for name in ('pipeline','utils','models','models.clap_encoder'): sys.modules.pop(name,None)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
import models.CLAP.open_clip.factory as clap_factory
_original_load_state_dict=clap_factory.load_state_dict
def _safe_clap_state_dict(path,map_location='cpu',skip_params=True):
    state=_original_load_state_dict(path,map_location=map_location,skip_params=skip_params)
    for k in [k for k in list(state) if k.endswith('embeddings.position_ids')]: state.pop(k,None)
    return state
clap_factory.load_state_dict=_safe_clap_state_dict
from pipeline import build_audiosep
t0=time.perf_counter()
with scoped_verified_torch_load({MODEL_CKPT,CLAP_CKPT}):
    model=build_audiosep(config_yaml=str(REPO/'config'/'audiosep_base.yaml'),checkpoint_path=str(MODEL_CKPT),device=torch.device('cuda'))
print('[OK] AudioSep loaded in',round(time.perf_counter()-t0,2),'s; audit:',AUDIT)


## Input recording
Upload a representative recording if possible (mono or stereo WAV/MP3/FLAC accepted by librosa). The model converts it to 32 kHz mono. If you skip upload, a deterministic synthetic mixture is generated for technical checks only.


In [ ]:
uploaded=files.upload()
if uploaded:
    import librosa
    input_name=next(iter(uploaded))
    y,sr=librosa.load(input_name,sr=32000,mono=True)
    SOURCE_LABEL=f'uploaded:{input_name}'
else:
    sr=32000; t=np.arange(32*32000,dtype=np.float32)/sr
    rng=np.random.default_rng(20261125)
    y=(0.22*np.sin(2*np.pi*440*t)+0.12*np.sin(2*np.pi*660*t)*(0.5+0.5*np.sin(2*np.pi*2*t))+0.035*rng.standard_normal(len(t))).astype(np.float32)
    SOURCE_LABEL='synthetic_tone_plus_noise_proxy'
y=np.asarray(y,dtype=np.float32).reshape(-1)
assert len(y)>=5*sr, 'Use an input at least 5 seconds long.'
print('Source:',SOURCE_LABEL,'| duration:',round(len(y)/sr,2),'s | peak:',float(np.max(np.abs(y))))


In [ ]:
def embed(query='ukulele'):
    with torch.no_grad():
        return model.query_encoder.get_query_embed(modality='text',text=[query],device='cuda')
COND=embed('ukulele')
def direct(x,cond=COND):
    inp={'mixture':torch.from_numpy(np.asarray(x,dtype=np.float32))[None,None,:].to('cuda'),'condition':cond.to('cuda')}
    with torch.inference_mode(): out=model.ss_model(inp)['waveform']
    return out.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)
def corr(a,b):
    a=np.asarray(a,dtype=np.float64); b=np.asarray(b,dtype=np.float64)
    if np.std(a)<1e-12 or np.std(b)<1e-12: return float('nan')
    return float(np.corrcoef(a,b)[0,1])
def compare(ref,out):
    n=min(len(ref),len(out)); a=np.asarray(ref[:n],np.float64); b=np.asarray(out[:n],np.float64)
    err=b-a; rms=lambda z: float(np.sqrt(np.mean(z*z))+1e-15)
    return {'correlation':corr(a,b),'relative_error':rms(err)/rms(a),'rmse':rms(err),'output_to_direct_rms_db':20*np.log10(rms(b)/rms(a)),'direct_rms':rms(a),'chunked_rms':rms(b)}

def chunked(x,core_s,context_s,cond=COND):
    x=np.asarray(x,dtype=np.float32).reshape(-1); n=len(x)
    core=max(1,int(round(core_s*sr))); ctx=max(0,int(round(context_s*sr)))
    out=np.zeros(n,dtype=np.float32)
    for start in range(0,n,core):
        stop=min(n,start+core); left=max(0,start-ctx); right=min(n,stop+ctx)
        seg=x[left:right]
        # At file boundaries, synthesize missing context by reflection where possible.
        lp=max(0,ctx-(start-left)); rp=max(0,ctx-(right-stop))
        if lp or rp:
            mode='reflect' if len(seg)>1 else 'constant'
            seg=np.pad(seg,(lp,rp),mode=mode)
        pred=direct(seg,cond)
        offset=start-left+lp
        core_pred=pred[offset:offset+(stop-start)]
        if len(core_pred)!=(stop-start): raise RuntimeError(f'Core length mismatch: {len(core_pred)} != {stop-start}')
        out[start:stop]=core_pred
    return out


## Experiment A — direct inference length / shape checks
Direct inference is run at several lengths. If the full 32-second input causes an out-of-memory error, reduce `DIRECT_LENGTHS_SECONDS` and rerun this cell. Do not silently substitute chunked output for a direct reference.


In [ ]:
DIRECT_LENGTHS_SECONDS=[3.5,5.0,7.5,13.5]
direct_refs={}; direct_rows=[]
for seconds in DIRECT_LENGTHS_SECONDS:
    xx=y[:int(seconds*sr)]
    if len(xx)<int(seconds*sr):
        xx=np.pad(xx,(0,int(seconds*sr)-len(xx)),mode='reflect')
    torch.cuda.empty_cache(); t0=time.perf_counter()
    pred=direct(xx)
    elapsed=time.perf_counter()-t0
    assert pred.dtype==np.float32 and pred.ndim==1 and len(pred)==len(xx)
    direct_refs[seconds]=pred
    direct_rows.append({'duration_s':seconds,'input_samples':len(xx),'output_samples':len(pred),'dtype':str(pred.dtype),'inference_s':round(elapsed,3),'output_rms':float(np.sqrt(np.mean(pred.astype(np.float64)**2)))})
    print(direct_rows[-1])
pd.DataFrame(direct_rows).to_csv(OUT/'direct_length_checks.csv',index=False)


## Experiment B — context/core sweep against direct full-clip inference

The 13.5-second direct output is the reference. Each setting independently processes the entire clip using fixed output cores with left/right context. This is intentionally a modest sweep to limit GPU memory and runtime. Add settings only after this initial run completes reliably.


In [ ]:
REF_SECONDS=13.5
if REF_SECONDS not in direct_refs: raise RuntimeError('Run Experiment A and ensure the 13.5s direct reference succeeded.')
x_ref=y[:int(REF_SECONDS*sr)]
if len(x_ref)<int(REF_SECONDS*sr): x_ref=np.pad(x_ref,(0,int(REF_SECONDS*sr)-len(x_ref)),mode='reflect')
ref=direct_refs[REF_SECONDS]
CORE_SECONDS=[1.5,3.0,6.0]
CONTEXT_SECONDS=[0.5,1.0,2.0,3.0]
rows=[]; outputs={}
for core_s in CORE_SECONDS:
  for context_s in CONTEXT_SECONDS:
    torch.cuda.empty_cache(); t0=time.perf_counter()
    try:
      out=chunked(x_ref,core_s,context_s)
      metrics=compare(ref,out); elapsed=time.perf_counter()-t0
      row={'core_s':core_s,'context_s':context_s,'chunks':int(np.ceil(len(x_ref)/(core_s*sr))),'elapsed_s':round(elapsed,3),**metrics,'status':'ok'}
      rows.append(row); outputs[(core_s,context_s)]=out
      print(row)
    except torch.cuda.OutOfMemoryError as exc:
      torch.cuda.empty_cache(); rows.append({'core_s':core_s,'context_s':context_s,'status':'oom','error':str(exc)[:300]}); print('OOM:',core_s,context_s)
    except Exception as exc:
      rows.append({'core_s':core_s,'context_s':context_s,'status':'error','error':repr(exc)[:500]}); print('ERROR:',core_s,context_s,repr(exc))
df=pd.DataFrame(rows); df.to_csv(OUT/'context_sweep_metrics.csv',index=False)


In [ ]:
ok=df[df.status=='ok'].copy().sort_values(['relative_error','rmse'])
display(ok[['core_s','context_s','chunks','correlation','relative_error','rmse','output_to_direct_rms_db','elapsed_s']])
if len(ok):
    pivot=ok.pivot(index='context_s',columns='core_s',values='correlation')
    ax=pivot.plot(kind='bar',figsize=(9,5)); ax.set_ylabel('Correlation vs direct'); ax.set_title('AudioSep chunk context/core sweep'); ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.savefig(OUT/'correlation_sweep.png',dpi=160); plt.show()
    pivot2=ok.pivot(index='context_s',columns='core_s',values='relative_error')
    ax=pivot2.plot(kind='bar',figsize=(9,5)); ax.set_ylabel('Relative RMS error vs direct'); ax.set_title('Chunked/direct relative error'); ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.savefig(OUT/'relative_error_sweep.png',dpi=160); plt.show()
best=ok.iloc[0].to_dict() if len(ok) else None
print('Best setting by lowest relative RMS error:',best)


## Experiment C — upstream chunker defect reproduction
This runs the pinned upstream `chunk_inference()` on a short input and compares it with direct inference. The expected issue is that the upstream loop does not run for inputs no longer than its 5-second window, leaving zeros. This is a source-behavior check, not a production candidate.


In [ ]:
short=y[:int(3.5*sr)]
inp={'mixture':torch.from_numpy(short)[None,None,:].to('cuda'),'condition':COND.to('cuda')}
with torch.no_grad(): upstream=model.ss_model.chunk_inference(inp)
up=np.asarray(upstream).reshape(-1)
print({'upstream_shape':tuple(np.asarray(upstream).shape),'upstream_dtype':str(np.asarray(upstream).dtype),'upstream_rms':float(np.sqrt(np.mean(up.astype(np.float64)**2))),'direct_rms':float(np.sqrt(np.mean(direct_refs[3.5].astype(np.float64)**2)))})


In [ ]:
summary={
 'source_label':SOURCE_LABEL,'sample_rate_hz':sr,'input_duration_s':len(y)/sr,
 'audiosep_source_commit':PIN,'checkpoint_sha256':{str(k.name):v for k,v in EXPECTED.items()},
 'direct_length_checks':direct_rows,'context_sweep_rows':rows,
 'best_by_relative_error':best,
 'interpretation_warning':'Direct inference is only a self-consistency reference, not clean ground truth. High correlation does not establish perceptual quality. Compare auditioned outputs and residuals before any integration decision.',
 'torch_load_audit':AUDIT,
}
(OUT/'context_sweep_summary.json').write_text(json.dumps(summary,indent=2,default=str))
if len(ok):
    best_key=(float(best['core_s']),float(best['context_s']))
    best_audio=outputs[best_key]
    sf.write(OUT/'best_chunked_vs_direct.wav',best_audio,sr,subtype='FLOAT')
    sf.write(OUT/'direct_reference.wav',ref,sr,subtype='FLOAT')
zip_path=shutil.make_archive(str(OUT/'audiosep_context_sweep_results'),'zip',root_dir=OUT)
print('Saved:',zip_path)
print('Drive folder:',OUT)


## Decision gate
- **Do not integrate** merely because one setting has a high correlation.
- Inspect the best and direct WAVs by listening, and check correlation, relative error, output level, chunk-boundary discontinuities, and memory/runtime.
- If the sweep remains unstable, investigate model normalization / temporal receptive field and the distinction between full-clip and crop inference. Do not keep tuning chunk boundaries without evidence.
- This experiment does not alter SONORA's app or Stage 4 DSP pipeline.
